# Classify difference between frog and cats


### Setup

In [1]:
import torch

In [2]:
import random
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision
from torchvision import transforms
from PIL import Image

In [3]:
# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### Transforms

In [6]:
img_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### Load Images from CIFAR10

In [13]:
from torch.utils.data import Dataset
import torchvision

class BinaryCatFrogCIFAR10(Dataset):
    """
    0 -> cat
    1 -> frog
    """
    def __init__(self, root="./data", train=True, transform=None, download=True):
        self.base = torchvision.datasets.CIFAR10(
            root=root,
            train=train,
            download=download
        )
        self.transform = transform

        classes = self.base.classes

        self.cat_idx = classes.index("cat")
        self.frog_idx = classes.index("frog")

        self.samples = []
        for i, target in enumerate(self.base.targets):
            if target == self.cat_idx:
                self.samples.append((i, 0))
            elif target == self.frog_idx:
                self.samples.append((i, 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base_idx, label = self.samples[idx]
        image, _ = self.base[base_idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [14]:
full_train_dataset = BinaryCatFrogCIFAR10(
    root="./data",
    train=True,
    transform=img_transforms,
    download=True
)

In [15]:
test_data = BinaryCatFrogCIFAR10(
    root="./data",
    train=False,
    transform=img_transforms,
    download=True
)

In [16]:
# Split train into train/val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_data, val_data = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

### Create DataLoader

In [17]:
batch_size=64
train_data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
val_data_loader  = torch.utils.data.DataLoader(val_data, batch_size=batch_size)
test_data_loader  = torch.utils.data.DataLoader(test_data, batch_size=batch_size)

### Create Model

SimpleNet is a model of three Linear layers and ReLu activations between them.

**Note**! No need for softmax() in the forward(), because crossEntropy loss add it. However, it is needed in the training function during the validation phase.

In [18]:
class SimpleNet(nn.Module):

    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(12288, 84)
        self.fc2 = nn.Linear(84, 50)
        self.fc3 = nn.Linear(50,2)

    def forward(self, x):
        x = x.view(-1, 12288) # change size of 3D Tensor to 1D
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [19]:
simplenet = SimpleNet()

### Create an Optimizer

In [20]:
optimizer = optim.Adam(simplenet.parameters(), lr=0.001)

### Copy the Model to GPU

In [21]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

simplenet.to(device)

SimpleNet(
  (fc1): Linear(in_features=12288, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=2, bias=True)
)

### Training

In [28]:
def train(model, optimizer, loss_fn, train_loader, val_loader, epochs=20, device="cpu"):
    for epoch in range(1, epochs+1):
        training_loss = 0.0
        valid_loss = 0.0
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            output = model(inputs)
            loss = loss_fn(output, targets)
            loss.backward()
            optimizer.step()
            training_loss += loss.data.item() * inputs.size(0)
        training_loss /= len(train_loader.dataset)

        model.eval()
        num_correct = 0
        num_examples = 0
        for batch in val_loader:
            inputs, targets = batch
            inputs = inputs.to(device)
            output = model(inputs)
            targets = targets.to(device)
            loss = loss_fn(output,targets)
            valid_loss += loss.data.item() * inputs.size(0)
            correct = torch.eq(torch.max(F.softmax(output, dim=1), dim=1)[1], targets)
            num_correct += torch.sum(correct).item()
            num_examples += correct.shape[0]
        valid_loss /= len(val_loader.dataset)

        print('Epoch: {}, Training Loss: {:.2f}, Validation Loss: {:.2f}, accuracy = {:.2f}'.format(epoch, training_loss,
        valid_loss, num_correct / num_examples))

In [29]:
train(simplenet, optimizer,torch.nn.CrossEntropyLoss(), train_data_loader,val_data_loader, epochs=5, device=device)

Epoch: 1, Training Loss: 0.43, Validation Loss: 0.50, accuracy = 0.76
Epoch: 2, Training Loss: 0.39, Validation Loss: 0.51, accuracy = 0.76
Epoch: 3, Training Loss: 0.36, Validation Loss: 0.54, accuracy = 0.76
Epoch: 4, Training Loss: 0.32, Validation Loss: 0.55, accuracy = 0.77
Epoch: 5, Training Loss: 0.30, Validation Loss: 0.60, accuracy = 0.76


### Making predictions

In [33]:
labels = ['cat', 'frog']

img, true_label = test_data[0]          # unpack tuple
img = img.to(device)                   # move to device
img = torch.unsqueeze(img, 0)          # add batch dim

simplenet.eval()
with torch.no_grad():
    prediction = simplenet(img).argmax(dim=1).item()

print("Predicted:", labels[prediction])
print("True:", labels[true_label])

Predicted: cat
True: cat


### Saving Models

Saves the entire model using save or just the parameters using state_dict. Using the latter is normally preferable, as it allows reusing parameters even if the model's structure changes (or apply parameters from one model to another).

In [36]:
# option 1 - not preffered
# torch.save(simplenet, "/tmp/simplenet")
# simplenet = torch.load("/tmp/simplenet")

In [40]:
# option 2 - preffered
torch.save(simplenet.state_dict(), "/tmp/simplenet")

In [41]:
simplenet = SimpleNet()
simplenet_state_dict = torch.load("/tmp/simplenet")
simplenet.load_state_dict(simplenet_state_dict)

<All keys matched successfully>